# WasteValue AI — เตรียมข้อมูล (โจทย์ข้อ 2)

notebook นี้ทำความสะอาดชุดข้อมูล **Garbage Classification (12 classes)** จาก Kaggle ให้พร้อมเทรน แล้วแบ่ง test และ 5 fold สำหรับ cross validation

**สิ่งที่ได้ออกมา** (เซฟลงโฟลเดอร์ `palm_output` ใน Google Drive)
| ไฟล์ | ส่งให้ | ใช้ทำอะไร |
|---|---|---|
| `split.csv` | แฟนต้า | รายชื่อไฟล์ภาพ + label (13 คลาส) + ประเภทขยะ + train/test + เลข fold |
| `class_weights.json` | แฟนต้า | น้ำหนักคลาส เผื่อใช้ตอนเทรน |
| `category_map.json` | แฟนต้า, A | แปลงคลาสเป็นประเภทขยะ (ใช้ประเมินผล 2 ระดับ และใช้ในเว็บ) |
| `cleaning_summary.csv` | สไลด์ | ตัวเลขก่อน/หลังแต่ละขั้น |
| `*.png` | สไลด์ | กราฟจำนวนภาพต่อคลาส และตัวอย่างภาพ |

**วิธีใช้:** อ่านคำอธิบายแต่ละขั้น แล้วกดรันทีละ cell จากบนลงล่าง (ไม่ต้องใช้ GPU)
ส่วนที่ต้องแก้มีแค่ cell **ตั้งค่า** ด้านล่าง กับ cell **ขั้นที่ 7** ที่ให้จดภาพที่ label ผิด

## ตั้งค่า
แก้เฉพาะ `DRIVE_FOLDER` ให้ตรงกับโฟลเดอร์ที่ทีมใช้ใน Google Drive

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/WasteValue"   # โฟลเดอร์ทีมใน Google Drive
ZIP_NAME     = "garbage-classification.zip"           # ไฟล์ zip ที่โหลดจาก Kaggle
EXTRACT_TO   = "/content/data"                        # แตกไฟล์ไว้ในเครื่อง Colab (เร็วกว่าใน Drive)
OUT_DIR      = DRIVE_FOLDER + "/palm_output"          # ที่เก็บผลลัพธ์

# โฟลเดอร์ 12 คลาสในชุดข้อมูลจาก Kaggle
SOURCE_CLASSES = ["battery", "biological", "brown-glass", "cardboard", "clothes", "green-glass",
                  "metal", "paper", "plastic", "shoes", "trash", "white-glass"]

# คลาสที่โมเดลเรียน 13 คลาส (trash แบ่งเป็น 2) ลำดับนี้ต้องตรงกันทั้งทีม
CLASSES = ["battery", "biological", "brown-glass", "cardboard", "clothes", "green-glass", "metal",
           "paper", "plastic", "shoes", "trash_general", "trash_infectious", "white-glass"]

# แปลงคลาสเป็นประเภทขยะ (ใช้ตอนประเมินผลระดับประเภท และใช้ในเว็บ)
CATEGORY_MAP = {
    "battery": "hazardous",          # ขยะอันตราย
    "biological": "organic",         # ขยะอินทรีย์
    "cardboard": "recyclable", "paper": "recyclable", "metal": "recyclable", "plastic": "recyclable",
    "white-glass": "recyclable", "brown-glass": "recyclable", "green-glass": "recyclable",   # ขยะรีไซเคิล
    "clothes": "general", "shoes": "general", "trash_general": "general",                   # ขยะทั่วไป
    "trash_infectious": "infectious",  # ขยะติดเชื้อ
}
assert set(CATEGORY_MAP) == set(CLASSES)
MAX_PER_CLASS = 1000   # คลาสไหนมีภาพเกินนี้ จะสุ่มเก็บไว้แค่นี้ (แก้ปัญหา clothes มีภาพเยอะเกิน)

TEST_SIZE = 0.15   # แยกเป็น test 15%
N_FOLDS   = 5      # ส่วนที่เหลือแบ่งเป็น 5 fold
SEED      = 42

In [ ]:
import os, glob, json, hashlib, random, zipfile, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("ไม่ได้รันบน Colab: ข้ามการเชื่อม Google Drive")
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
summary = []   # เก็บตัวเลขแต่ละขั้นไว้ทำสไลด์

## ขั้นที่ 1: โหลดข้อมูล

ทำ **อย่างใดอย่างหนึ่ง** ก่อนรัน cell นี้:
- **วิธี A (ง่ายสุด):** เข้า kaggle.com ค้น *Garbage Classification 12 classes* (ของ mostafaabla) → กด Download → อัปโหลดไฟล์ zip ไว้ใน `DRIVE_FOLDER` แล้วตั้งชื่อเป็น `garbage-classification.zip`
- **วิธี B:** ที่ Kaggle ไปหน้า Settings → API → Create New Token จะได้ไฟล์ `kaggle.json` อัปโหลดไว้ใน `DRIVE_FOLDER` แล้ว cell นี้จะโหลดให้เอง

แนะนำให้แชร์โฟลเดอร์ `DRIVE_FOLDER` ให้แฟนต้าด้วย จะได้ใช้ zip ไฟล์เดียวกัน

In [ ]:
zip_path = os.path.join(DRIVE_FOLDER, ZIP_NAME)
kaggle_json = os.path.join(DRIVE_FOLDER, "kaggle.json")

if os.path.isdir(EXTRACT_TO) and os.listdir(EXTRACT_TO):
    print("แตกไฟล์ไว้แล้ว ข้ามขั้นนี้")
else:
    if not os.path.exists(zip_path):
        if os.path.exists(kaggle_json):
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            shutil.copy(kaggle_json, os.path.expanduser("~/.kaggle/kaggle.json"))
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
            os.system(f"kaggle datasets download -d mostafaabla/garbage-classification -p '{DRIVE_FOLDER}'")
            dl = glob.glob(os.path.join(DRIVE_FOLDER, "garbage-classification*.zip"))
            if dl and dl[0] != zip_path:
                os.rename(dl[0], zip_path)
        else:
            raise FileNotFoundError(f"ไม่เจอ {zip_path} หรือ {kaggle_json} ดูวิธี A/B ด้านบน")
    print("กำลังแตกไฟล์ (ประมาณ 1–2 นาที)...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(EXTRACT_TO)

# หาโฟลเดอร์ที่มีโฟลเดอร์ย่อยเป็นชื่อคลาส
DATA_ROOT = None
for root, dirs, _ in os.walk(EXTRACT_TO):
    if {"battery", "plastic", "paper"}.issubset(set(dirs)):
        DATA_ROOT = root
        break
assert DATA_ROOT, "หาโฟลเดอร์คลาสไม่เจอ ลองเปิดดูโครงสร้างใน EXTRACT_TO"
print("DATA_ROOT =", DATA_ROOT)
print("โฟลเดอร์ที่เจอ:", sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))))

## ขั้นที่ 2: สำรวจข้อมูลดิบ
นับจำนวนไฟล์ทุกคลาสก่อนทำอะไร

In [ ]:
rows = []
for cls in sorted(os.listdir(DATA_ROOT)):
    d = os.path.join(DATA_ROOT, cls)
    if not os.path.isdir(d):
        continue
    for f in sorted(os.listdir(d)):
        if os.path.isfile(os.path.join(d, f)):
            rows.append({"path": f"{cls}/{f}", "label": cls})
df = pd.DataFrame(rows)
summary.append({"ขั้น": "ข้อมูลดิบ", "จำนวนภาพ": len(df)})

raw_counts = df["label"].value_counts().sort_index()
print(raw_counts.to_string()); print("รวม", len(df), "ไฟล์")
ax = raw_counts.plot(kind="bar", figsize=(10, 4), color="#9AA8A0", title="Images per class (raw)")
ax.set_ylabel("images"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_class_counts_raw.png", dpi=150); plt.show()

## ขั้นที่ 3: ตรวจชื่อโฟลเดอร์
เช็กว่าโฟลเดอร์ในข้อมูลตรงกับ 12 คลาสของชุดข้อมูล ไม่มีเกิน ไม่มีขาด

In [ ]:
found = set(df["label"])
assert found == set(SOURCE_CLASSES), f"โฟลเดอร์ไม่ตรง | เกิน: {found - set(SOURCE_CLASSES)} | ขาด: {set(SOURCE_CLASSES) - found}"
print(f"ครบ {len(SOURCE_CLASSES)} โฟลเดอร์ ตรงกับชุดข้อมูล (trash จะถูกแบ่งเป็น 2 คลาสในขั้นที่ 7.5)")

## ขั้นที่ 4: ตรวจภาพเสีย (ข้อมูลขาดหาย)
เปิดทุกไฟล์ ถ้าเปิดไม่ได้ ไฟล์ว่าง หรือไม่ใช่ภาพ จะถูกตัดทิ้ง พร้อมเก็บขนาดและโหมดสี (RGB, ขาวดำ, โปร่งใส) ไว้ดูในขั้นที่ 6

In [ ]:
def inspect(rel):
    p = os.path.join(DATA_ROOT, rel)
    try:
        if os.path.getsize(p) == 0:
            return None
        with Image.open(p) as im:
            im.verify()                      # เช็กโครงสร้างไฟล์
        with Image.open(p) as im:
            im.load()                        # เช็กว่าอ่านพิกเซลได้จริง
            return im.mode, im.width, im.height
    except Exception:
        return None

info = []
for i, rel in enumerate(df["path"]):
    info.append(inspect(rel))
    if (i + 1) % 2000 == 0:
        print(f"ตรวจแล้ว {i + 1}/{len(df)}")
bad_mask = [x is None for x in info]
bad = df[bad_mask]
print(f"\nภาพเสียหรือเปิดไม่ได้: {len(bad)} ภาพ")
if len(bad): print(bad.to_string())

df = df[[not b for b in bad_mask]].reset_index(drop=True)
ok = [x for x in info if x is not None]
df["mode"]   = [x[0] for x in ok]
df["width"]  = [x[1] for x in ok]
df["height"] = [x[2] for x in ok]
summary.append({"ขั้น": "ตัดภาพเสีย", "จำนวนภาพ": len(df)})

## ขั้นที่ 5: ตรวจภาพซ้ำ
ถ้าภาพเดียวกันไปอยู่ทั้งใน train และ test โมเดลจะเหมือนเคยเห็นข้อสอบมาก่อน ผลจะดูดีเกินจริง (**data leakage**)

ตรวจ 2 แบบ:
- **ซ้ำเป๊ะ:** ไฟล์เหมือนกันทุก byte (MD5 hash)
- **ซ้ำแทบเหมือน:** ภาพเดียวกันแต่ย่อขนาดหรือบีบอัดต่างกัน (difference hash จากภาพย่อ 9×8)

ถ้าภาพซ้ำอยู่คลาสเดียวกัน เก็บไว้ 1 ภาพ ถ้าซ้ำกัน**แต่ label ต่างกัน** ตัดทิ้งทั้งหมด เพราะไม่รู้ว่าอันไหนถูก

In [ ]:
def md5(rel):
    with open(os.path.join(DATA_ROOT, rel), "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dhash(rel, size=8):
    with Image.open(os.path.join(DATA_ROOT, rel)) as im:
        g = np.asarray(im.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return "".join("1" if b else "0" for b in bits)

df["md5"]   = [md5(r) for r in df["path"]]
df["dhash"] = [dhash(r) for r in df["path"]]

def dedup(df, key, name):
    groups = df.groupby(key)["label"].agg(["count", "nunique"])
    conflict_keys = groups[groups["nunique"] > 1].index
    conflict = df[df[key].isin(conflict_keys)]
    keep = df[~df[key].isin(conflict_keys)].drop_duplicates(subset=key, keep="first")
    removed_same = len(df) - len(conflict) - len(keep)
    print(f"[{name}] ซ้ำในคลาสเดียวกัน ตัด {removed_same} ภาพ | ซ้ำแต่ label ต่างกัน ตัด {len(conflict)} ภาพ")
    return keep.reset_index(drop=True), conflict

dup_examples = df[df.duplicated("dhash", keep=False)].sort_values("dhash")
df, conflict1 = dedup(df, "md5", "ซ้ำเป๊ะ")
df, conflict2 = dedup(df, "dhash", "ซ้ำแทบเหมือน")
conflicts = pd.concat([conflict1, conflict2])
if len(conflicts):
    print("\nตัวอย่างภาพที่ซ้ำแต่ label ต่างกัน:")
    print(conflicts[["path", "label"]].head(20).to_string())
summary.append({"ขั้น": "ตัดภาพซ้ำ", "จำนวนภาพ": len(df)})

In [ ]:
# ดูตัวอย่างคู่ภาพซ้ำ (เอาไปใส่สไลด์ได้)
pairs = [g for _, g in dup_examples.groupby("dhash") if len(g) >= 2][:4]
if pairs:
    fig, axes = plt.subplots(len(pairs), 2, figsize=(6, 3 * len(pairs)), squeeze=False)
    for row, g in zip(axes, pairs):
        for ax, (_, r) in zip(row, g.head(2).iterrows()):
            ax.imshow(Image.open(os.path.join(DATA_ROOT, r["path"])).convert("RGB"))
            ax.set_title(r["path"], fontsize=8); ax.axis("off")
    plt.suptitle("Duplicate image examples"); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/02_duplicate_examples.png", dpi=150); plt.show()
else:
    print("ไม่พบภาพซ้ำ")

## ขั้นที่ 6: รูปแบบภาพ
ดูว่ามีภาพขาวดำ (`L`) ภาพโปร่งใส (`RGBA`, `P`) และขนาดภาพต่างกันแค่ไหน

ขั้นนี้**ไม่ตัดภาพทิ้ง** แค่บันทึกไว้ เพราะตอนเทรน notebook ของแฟนต้าจะแปลงทุกภาพเป็น RGB และย่อเป็น 224×224 ให้เท่ากันอยู่แล้ว

In [ ]:
print("โหมดสีของภาพ:")
print(df["mode"].value_counts().to_string())
print("\nขนาดภาพ (พิกเซล):")
print(df[["width", "height"]].describe().round(0).to_string())

## ขั้นที่ 7: ตรวจ label ด้วยตา
cell ด้านล่างสุ่มโชว์คลาสละ 20 ภาพ ตัวเลขบนภาพคือ **id**

คลาส `trash` ไม่ต้องตรวจในขั้นนี้ เพราะจะคัดทุกภาพในขั้นที่ 7.5

ไล่ดูว่ามีภาพที่อยู่ผิดคลาส หรือไม่ใช่ขยะเลย (เช่น ภาพกราฟิก ภาพที่มีหลายอย่างปนกัน) ถ้าเจอ ให้จด id ใส่ `REMOVE_IDS` ใน cell ถัดไป

ไม่ต้องไล่ทุกภาพ ใช้แค่เป็นการสุ่มตรวจ ในรายงานเขียนได้ว่า "สุ่มตรวจ 200 ภาพ พบผิด x ภาพ" 

In [ ]:
df["id"] = range(len(df))
SHOW_PER_CLASS = 20
for cls in SOURCE_CLASSES:
    sub = df[df["label"] == cls]
    sample = sub.sample(min(SHOW_PER_CLASS, len(sub)), random_state=SEED)
    fig, axes = plt.subplots(2, 10, figsize=(20, 4.6))
    for ax in axes.flat: ax.axis("off")
    for ax, (_, r) in zip(axes.flat, sample.iterrows()):
        ax.imshow(Image.open(os.path.join(DATA_ROOT, r["path"])).convert("RGB"))
        ax.set_title(f"id {r['id']}", fontsize=9)
    fig.suptitle(f"{cls}  ({len(sub)} images)", fontsize=14, x=0.02, ha="left")
    plt.tight_layout(); plt.show()

In [ ]:
REMOVE_IDS = []   # ใส่ id ที่ label ผิด เช่น [123, 4567]

checked = SHOW_PER_CLASS * len(SOURCE_CLASSES)
df = df[~df["id"].isin(REMOVE_IDS)].reset_index(drop=True)
print(f"สุ่มตรวจ {checked} ภาพ พบ label ผิด {len(REMOVE_IDS)} ภาพ (ตัดออกแล้ว)")
summary.append({"ขั้น": "ตัดภาพ label ผิด (สุ่มตรวจ)", "จำนวนภาพ": len(df)})

## ขั้นที่ 7.5: แบ่งคลาส trash เป็น 2 คลาส
คลาส `trash` มีขยะ 2 แบบปนกัน จึงแบ่งเป็น:
- `trash_infectious` ขยะติดเชื้อ เช่น หน้ากากอนามัย ผ้าอนามัย แพมเพิส
- `trash_general` ขยะทั่วไป เช่น แปรงสีฟัน

วิธีทำ:
1. เปิดเครื่องมือ **คัดภาพคลาส Trash** (ลิงก์จากทีม) เลือกโฟลเดอร์ `trash` ในเครื่อง แล้วกด 1 / 2 / 3 ทีละภาพ
2. คัดเสร็จ กด **คัดลอกผลลัพธ์** แล้ววางในบรรทัดว่างระหว่างเครื่องหมายคำพูดสามตัว ใน cell ด้านล่าง

ผลการคัด: `infectious` → trash_infectious, `general` → trash_general, `remove` → ตัดทิ้ง
**ควรคัดให้ครบทุกภาพ** ภาพที่ไม่ได้คัดจะถูกนับเป็น trash_infectious ไปก่อน

In [ ]:
TRASH_LABELS = """
"""

lines = [l.strip() for l in TRASH_LABELS.strip().splitlines() if l.strip() and not l.lower().startswith("filename")]
decided = {}
for l in lines:
    name, _, d = l.rpartition(",")
    if d not in ("infectious", "general", "remove"):
        raise ValueError(f"บรรทัดนี้อ่านไม่ได้: {l}")
    decided[name] = d

is_trash = df["label"] == "trash"
names = df["path"].str.split("/", n=1).str[1]
dec = names.where(is_trash).map(decided)

n_todo = int((is_trash & dec.isna()).sum())
if not decided:
    print("⚠️ ยังไม่ได้วางผลการคัด: ภาพ trash ทั้งหมดจะเป็น trash_infectious และจะไม่มีคลาส trash_general")
elif n_todo:
    print(f"⚠️ ยังไม่ได้คัด {n_todo} ภาพ: นับเป็น trash_infectious ไปก่อน")

df.loc[is_trash & (dec == "general"), "label"] = "trash_general"
df.loc[is_trash & (dec != "general") & (dec != "remove"), "label"] = "trash_infectious"
removed = int((is_trash & (dec == "remove")).sum())
df = df[~(is_trash & (dec == "remove"))].reset_index(drop=True)

not_found = set(decided) - set(names[is_trash])
if not_found:
    print(f"หมายเหตุ: {len(not_found)} ไฟล์ในผลการคัดไม่อยู่ในข้อมูลแล้ว (อาจถูกตัดตอนตัดภาพซ้ำ) ไม่เป็นไร")
print(f"trash_infectious {int((df.label == 'trash_infectious').sum())} ภาพ | "
      f"trash_general {int((df.label == 'trash_general').sum())} ภาพ | ลบทิ้ง {removed} ภาพ")
if 0 < (df.label == "trash_general").sum() < 100:
    print("⚠️ trash_general มีภาพน้อยกว่า 100 โมเดลอาจเรียนคลาสนี้ได้ไม่ดี ให้บอกทีม")
summary.append({"ขั้น": "แบ่ง trash เป็น 2 คลาส (ลบภาพที่ใช้ไม่ได้)", "จำนวนภาพ": len(df)})

## ขั้นที่ 8: ความสมดุลของคลาส
ถ้าบางคลาสมีภาพมากกว่าคลาสอื่นมาก โมเดลจะเอนไปทายคลาสนั้นบ่อย

ชุดนี้คลาส `clothes` (เสื้อผ้า) มีภาพมากกว่าคลาสอื่นหลายเท่า จึง**สุ่มเก็บไว้ไม่เกิน `MAX_PER_CLASS` ภาพต่อคลาส** (undersampling) แล้วดู **อัตราส่วน มากสุด : น้อยสุด** ถ้าไม่เกินราว 2:1 ถือว่ายังพอรับได้

ขั้นนี้ยังคำนวณ **class weights** เก็บไว้ให้แฟนต้าเลือกใช้ตอนเทรน (ให้น้ำหนักคลาสที่มีภาพน้อยมากขึ้น) และแนะนำให้ใช้ **macro F1** เป็นตัวชี้วัดหลัก

In [ ]:
before_counts = df["label"].value_counts().reindex(CLASSES)
print("ก่อนจำกัดจำนวน:"); print(before_counts.to_string())
df = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED)
                for _, g in df.groupby("label")]).reset_index(drop=True)
summary.append({"ขั้น": f"จำกัดไม่เกิน {MAX_PER_CLASS} ภาพต่อคลาส", "จำนวนภาพ": len(df)})

counts = df["label"].value_counts().reindex(CLASSES)
print("\nหลังจำกัดจำนวน:")
ratio = counts.max() / counts.min()
print(counts.to_string())
print(f"\nมากสุด : น้อยสุด = {counts.max()} : {counts.min()} = {ratio:.2f} : 1")

ax = counts.plot(kind="bar", figsize=(10, 4), color="#1F7A4D", title="Images per class (after cleaning)")
ax.axhline(counts.mean(), ls="--", color="gray", label=f"mean = {counts.mean():.0f}")
ax.set_ylabel("images"); ax.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_class_counts_clean.png", dpi=150); plt.show()

## ขั้นที่ 9: แบ่ง test และ 5-fold
ทำตาม Model Evaluation ที่เรียน:
1. แยก **test 15%** ออกก่อน (stratified) เก็บไว้วัดผลครั้งสุดท้ายครั้งเดียว ห้ามใช้ปรับโมเดล
2. ส่วน **train 85%** ที่เหลือ แบ่งเป็น **5 fold** แบบ stratified (ทุก fold มีสัดส่วนคลาสเท่ากัน) แล้วเขียนเลข fold 0–4 ไว้ให้ทุกภาพ

แฟนต้าเลือกได้ 2 แบบจากไฟล์เดียวกัน:
- **5-fold cross validation:** เทรน 5 รอบ แต่ละรอบใช้ 1 fold เป็น validation อีก 4 fold เป็น train แล้วเฉลี่ยคะแนน
- **แบบเร็ว:** เทรนรอบเดียว ใช้ fold 0 เป็น validation

augmentation ทำเฉพาะภาพที่ใช้เทรน ตอนเทรน ไม่ทำกับ validation และ test

In [ ]:
from sklearn.model_selection import StratifiedKFold

train_df, test_df = train_test_split(df, test_size=TEST_SIZE, stratify=df["label"], random_state=SEED)
train_df, test_df = train_df.copy(), test_df.copy()
train_df["split"], test_df["split"] = "train", "test"
train_df["fold"], test_df["fold"] = -1, -1

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, idx) in enumerate(skf.split(train_df, train_df["label"])):
    train_df.iloc[idx, train_df.columns.get_loc("fold")] = k

final = pd.concat([train_df, test_df]).reset_index(drop=True)
final["label_idx"] = final["label"].map({c: i for i, c in enumerate(CLASSES)})
final["category"] = final["label"].map(CATEGORY_MAP)
assert final["label_idx"].notna().all() and final["category"].notna().all()
assert (final.loc[final.split == "train", "fold"] >= 0).all()

# ตรวจซ้ำว่าไม่มีภาพเดียวกันอยู่ทั้ง train และ test
overlap = set(final.loc[final.split == "train", "dhash"]) & set(final.loc[final.split == "test", "dhash"])
assert not overlap, "มีภาพซ้ำระหว่าง train กับ test"
print("ตรวจแล้ว: ไม่มีภาพซ้ำระหว่าง train กับ test")

final["group"] = np.where(final["split"] == "test", "test", "fold " + final["fold"].astype(str))
cols = [f"fold {k}" for k in range(N_FOLDS)] + ["test"]
split_table = pd.crosstab(final["label"], final["group"]).reindex(CLASSES)[cols]
split_table.loc["รวม"] = split_table.sum()
print(split_table.to_string())

colors = ["#14432F", "#1F7A4D", "#34B36B", "#7BCB8E", "#BFDCCB", "#F4B66A"]
ax = pd.crosstab(final["label"], final["group"]).reindex(CLASSES)[cols].plot(
    kind="bar", stacked=True, figsize=(11, 4), color=colors, title="5 folds (train) + test per class")
ax.set_ylabel("images"); ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_split_per_class.png", dpi=150); plt.show()

## ขั้นที่ 10: บันทึกผลลัพธ์

In [ ]:
final[["path", "label", "label_idx", "category", "split", "fold"]].to_csv(f"{OUT_DIR}/split.csv", index=False)

train_counts = final[final.split == "train"]["label"].value_counts().reindex(CLASSES)
weights = train_counts.sum() / (len(CLASSES) * train_counts)   # คลาสที่ภาพน้อยได้น้ำหนักมาก
with open(f"{OUT_DIR}/class_weights.json", "w") as f:
    json.dump({"classes": CLASSES, "weights": [round(float(0 if np.isnan(w) else w), 4) for w in weights]}, f, indent=2)
with open(f"{OUT_DIR}/category_map.json", "w") as f:
    json.dump(CATEGORY_MAP, f, indent=2)

summary_df = pd.DataFrame(summary)
summary_df["ตัดออก"] = (-summary_df["จำนวนภาพ"].diff()).fillna(0).astype(int)
summary_df.to_csv(f"{OUT_DIR}/cleaning_summary.csv", index=False, encoding="utf-8-sig")
print(summary_df.to_string(index=False))
print(f"\ntrain {len(train_df)} (แบ่ง {N_FOLDS} fold) | test {len(test_df)}")
print(f"\nบันทึกไฟล์ไว้ที่ {OUT_DIR}:")
for f in sorted(os.listdir(OUT_DIR)): print(" -", f)

## ส่งต่อ
- ส่ง `split.csv`, `class_weights.json`, `category_map.json` ให้แฟนต้า (หรือแชร์โฟลเดอร์ `palm_output`) ใน `split.csv` คอลัมน์ `fold` เป็น 0–4 สำหรับภาพ train และเป็น -1 สำหรับภาพ test
- แฟนต้าต้องแตก zip ไฟล์เดียวกันในเครื่อง Colab ของตัวเอง แล้วอ่าน path จาก `split.csv` ต่อจาก `DATA_ROOT`
- ภาพ `.png` และ `cleaning_summary.csv` ใช้ทำสไลด์ข้อ 2

## เผื่ออาจารย์ถาม
- **ทำไมต้องตัดภาพซ้ำ** ถ้าภาพเดียวกันอยู่ทั้ง train และ test ผลวัดจะสูงเกินจริง เพราะโมเดลจำภาพได้ ไม่ได้เรียนรู้ลักษณะของวัสดุ
- **ทำไมแบ่งแบบ stratified** ให้ทุกชุดมีสัดส่วนคลาสเหมือนกัน ผลวัดของ test จึงสะท้อนข้อมูลทั้งหมด
- **ทำไมต้องแยก test ออกก่อน** test ใช้วัดผลครั้งสุดท้ายครั้งเดียว ถ้าเอามาช่วยเลือกโมเดล ผล test จะไม่เป็นกลางแล้ว
- **ทำไมใช้ 5-fold cross validation** ถ้าใช้ validation ชุดเดียว คะแนนอาจสูงหรือต่ำเพราะบังเอิญได้ชุดง่ายหรือยาก การหมุนให้ทุก fold ได้เป็น validation แล้วเฉลี่ย ทำให้เปรียบเทียบการทดลองได้น่าเชื่อถือกว่า
- **ทำไม augmentation ทำแค่ train** test ต้องเป็นภาพจริงที่ไม่ถูกดัดแปลง เพื่อวัดผลกับข้อมูลแบบที่ระบบจะเจอจริง
- **ทำไมแบ่ง trash เป็น 2 คลาส** ในโฟลเดอร์ trash มีทั้งขยะติดเชื้อ (หน้ากาก แพมเพิส) และขยะทั่วไป (แปรงสีฟัน) ซึ่งต้องทิ้งต่างกัน ถ้าใช้ label เดียว ระบบจะบอกว่าแปรงสีฟันเป็นขยะติดเชื้อ
- **ทำไมไม่รวมเป็น 5 คลาสตามประเภทขยะ** ถ้าโมเดลรู้แค่ "รีไซเคิล" จะบอกวัสดุและราคาไม่ได้ จึงให้โมเดลเรียนคลาสละเอียด แล้วแปลงเป็นประเภทขยะด้วยตาราง ประเมินผลได้ทั้ง 2 ระดับ
- **ทำไมเก็บ clothes / shoes ไว้** โมเดลตอบได้เฉพาะคลาสที่เคยเรียน ถ้าตัดทิ้ง เวลามีคนอัปโหลดรูปเสื้อผ้า โมเดลจะฝืนทายเป็นคลาสอื่น เช่น กระดาษ แล้วระบบจะบอกว่าขายได้ ซึ่งผิด
- **ทำไมจำกัดจำนวนภาพต่อคลาส** clothes มีภาพมากกว่าคลาสอื่นหลายเท่า ถ้าไม่จำกัด โมเดลจะเอนไปทาย clothes บ่อยเกินจริง